In [ ]:
import math
import numpy as np 
from numpy.linalg import inv
import matplotlib.pyplot as plt
import openpyxl
import cmath
import graphviz
from array import array

import qiskit as q
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit.visualization import *
from qiskit.quantum_info import Pauli, SparsePauliOp, Operator
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import *
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator, SamplerV2 as Sampler

In [ ]:
# Save an IBM Quantum account and set it as your default account.
API_Token = '13d9540e280b61b7a4254fcdb05516180957df07523dfef1924f7d9363699a51826b42e7a063b3fb685f295f6d27f8ff61810a97f503a704c07bc959d6cc6e3f'
QiskitRuntimeService.save_account(
    channel = "ibm_quantum",
    
    instance ="ibm-q-hub-ntu/ntu-internal/default",
    token = API_Token,
    set_as_default = True,
    # Use `overwrite=True` if you're updating your token.
    overwrite = True,
)

# Load saved credentials
service = QiskitRuntimeService()

#For real Device
#backend = service.least_busy(operational=True, simulator=False)
backend_nazca = service.backend("ibm_nazca")
noise_model = NoiseModel.from_backend(backend_nazca)

# Get basis gates from noise model
basis_gates = noise_model.basis_gates

#For FakeSimulator
#backend = FakeManilaV2()

# Get coupling map from backend
coupling_map = [[0, 1], [1, 2], [3, 2], [3, 4]]

#For Aer
backend = AerSimulator(noise_model=noise_model,
                       coupling_map=coupling_map,
                       basis_gates=basis_gates)
#backend = AerSimulator()
#backend = AerSimulator.from_backend(backend_nazca)

backendqubitNum = backend.num_qubits

In [ ]:
#Build Four Initial State |0>, |1>, |0>+|1>, |0>-i|1>
qreg = QuantumRegister(2, 'q')
creg = ClassicalRegister(2, 'c')
q0_init = {}
q1_init = {}

for i in range (2):
    #rho0 state |0>
    InitCirZero = QuantumCircuit(qreg, creg)
    #rho1 state |1>
    InitCirOne = QuantumCircuit(qreg, creg)
    InitCirOne.x(i)
    #rho2 state |0>+|1>
    InitCirPlus = QuantumCircuit(qreg, creg)
    InitCirPlus.h(i)
    #rho3 state |0>+i|1>
    InitCirRight = QuantumCircuit(qreg, creg)
    InitCirRight.h(i)
    InitCirRight.s(i)
    
    if i == 0:
        q0_init = {'q0zero':InitCirZero, 'q0one':InitCirOne, 'q0plus':InitCirPlus, 'q0right':InitCirRight}
    else:
        q1_init = {'q1zero':InitCirZero, 'q1one':InitCirOne, 'q1plus':InitCirPlus, 'q1right':InitCirRight}

# Store the initial circuits in a dictionary
InitialState = {}
for q0, q0state in q0_init.items():
    for q1, q1state in q1_init.items():
        initcomb = q.circuit.QuantumCircuit.compose(q0state, q1state)
        InitialState.update({f'{q0}_{q1}':initcomb})

In [ ]:
#Build Three measurement X, Y, Z
q0_meas = {}
q1_meas = {}

for i in range (2):
    #I-measurement
    MeasCirI = QuantumCircuit(qreg, creg)
    #X-measurement
    MeasCirX = QuantumCircuit(qreg, creg)
    MeasCirX.h(i)
    MeasCirX.measure(i, i)
    #Y-measurement
    MeasCirY = QuantumCircuit(qreg, creg)
    MeasCirY.sdg(i)
    MeasCirY.h(i)
    MeasCirY.measure(i, i)
    #Z-measurement
    MeasCirZ = QuantumCircuit(qreg, creg)
    MeasCirZ.measure(i, i)
    
    if i == 0:
        q0_meas = {'q0Imeas':MeasCirI, 'q0Xmeas':MeasCirX, 'q0Ymeas':MeasCirY, 'q0Zmeas':MeasCirZ}
    else:
        q1_meas = {'q1Imeas':MeasCirI, 'q1Xmeas':MeasCirX, 'q1Ymeas':MeasCirY, 'q1Zmeas':MeasCirZ}
    
# Store the measurement circuits in a dictionary
Measurement = {}
for q0, q0meas in q0_meas.items():
    for q1, q1meas in q1_meas.items():
        meascomb = q.circuit.QuantumCircuit.compose(q0meas, q1meas)
        Measurement.update({f'{q0}_{q1}':meascomb})

In [ ]:
#Build each set of circuits with four kinds of Initialstates and four kinds of measurements
Circuit = {}
for InitName, Initial in InitialState.items():
    for MeasName, Measure in Measurement.items():
        circuit = q.circuit.QuantumCircuit.compose(Initial, Measure)
        circuit.measure_all()
        Circuit.update({f'{InitName}_{MeasName}':circuit})

In [ ]:
len(Circuit)

In [ ]:
shots = 1000
sampler = Sampler(mode=backend)

job = {}
for Name, Cir in Circuit.items():
    CirTran = q.compiler.transpile(Cir, backend=backend, optimization_level=0)
    job.update({Name:sampler.run([CirTran], shots=shots)})

In [ ]:
result = []
for index, job_name in job.items():
    res = job_name.result()
    result.append(res[0].data.meas.get_counts())
result

In [ ]:
qubit_zero = {}
qubit_one = {}
for i in range(len(result)):
    q_0sum_zero = 0
    q_0sum_one = 0
    q_1sum_zero = 0
    q_1sum_one = 0
    for k ,v in result[i].items():
        q_0allzero = 0
        q_0allone = 0
        if k[0] == '0':
            q_0sum_zero += v
            zero = 2*(int(q_0sum_zero)/shots) - 1
            q_0allzero = zero
            qubit_zero.update({f'0_{i}':zero})
        else:
            q_0sum_one += v
            one = 2*(int(q_0sum_one)/shots) - 1
            q_0allone = one
            qubit_zero.update({f'1_{i}':one})

        if q_0allzero == 1:
            qubit_zero.update({f'1_{i}':0})
        if q_0allone == 1:
            qubit_zero.update({f'0_{i}':0})

        q_1allzero = 0
        q_1allone = 0
        if k[1] == '0':
            q_1sum_zero += v
            zero = 2*(int(q_1sum_zero)/shots) - 1
            q_1allzero = zero
            qubit_one.update({f'0_{i}':zero})
        else:
            q_1sum_one += v
            one = 2*(int(q_1sum_one)/shots) - 1
            q_1allone = one
            qubit_one.update({f'1_{i}':one})

        if q_1allzero == 1:
            qubit_one.update({f'1_{i}':0})
        if q_1allone == 1:
            qubit_one.update({f'0_{i}':0})
    
for i in range (len(Circuit)):
    print(qubit_zero[f'0_{i}'])
print("\n")
for i in range (len(Circuit)):
    print(qubit_one[f'0_{i}'])

In [ ]:
#Create GramMatrix g
g = np.ones((4, 4))
row = 1
column = 0
for index, job_name in job.items():
    if row == 4:
        row = 1
        column += 1
    result = job_name.result()
    g[row][column] = result[0].data.evs
    print(index, "Expectation:", result[0].data.evs)
    row += 1

#Create State Preparation Matrix A
A = np.array([[1, 1, 1, 1],
              [0, 0, 1, 0],
              [0, 0, 0, 1],
              [1,-1, 0, 0]])

#Calculate Readout Matrix by the quation B = g * A^-1
A_inv = inv(A)
print(A_inv)
B = np.matmul(g, A_inv)
print(B)

#Calculate observable X, Y, Z
a_x = np.array([[0, 1, 0, 0]])
a_y = np.array([[0, 0, 1, 0]])
a_z = np.array([[0, 0, 0, 1]])

B_inv = inv(B)
q_x = np.matmul(a_x, B_inv)
q_y = np.matmul(a_y, B_inv)
q_z = np.matmul(a_z, B_inv)
print(B_inv, '\n\n', q_x, '\n\n', q_y, '\n\n', q_z)

In [ ]:
q_reg = QuantumRegister(1, 'q')
c_reg = ClassicalRegister(1, 'c')

testCir = QuantumCircuit(q_reg, c_reg)
testCir.h(0)
testCir.ry(-np.pi/8, 0)

matrix = np.array([[ 1/np.sqrt(2), -1/np.sqrt(2)],
        		   [-1/np.sqrt(2), -1/np.sqrt(2)]]) 
ErrorObservable = SparsePauliOp([('I' * (backendqubitNum-1) + 'X'), ('I' * (backendqubitNum-1) + 'Z')], coeffs=[-1/np.sqrt(2), 1/np.sqrt(2)])

In [ ]:
q_reg = QuantumRegister(1, 'q')
c_reg = ClassicalRegister(1, 'c')

testCir = QuantumCircuit(q_reg, c_reg)
testCir.h(0)
testCir.ry(-np.pi/8, 0)

matrix = np.array([[ 1/np.sqrt(2), -1/np.sqrt(2)],
        		   [-1/np.sqrt(2), -1/np.sqrt(2)]]) 
IdealObservable = SparsePauliOp([('I' * (backendqubitNum-1) + 'X'), ('I' * (backendqubitNum-1) + 'Y'), ('I' * (backendqubitNum-1) + 'Z')], coeffs=[(q_x[0][0]+q_z[0][0]), (q_x[0][1]+q_z[0][1]), (q_x[0][2]+q_z[0][2])])

In [ ]:
ErrorExpectation = []
for i in range (300):
    CirTran = q.compiler.transpile(testCir, backend=backend, optimization_level=0)
    job = estimator.run([(CirTran, ErrorObservable)])
    result = job.result()
    ErrorExpectation.append(result[0].data.evs)
    #print(f'Expectation_{i + 1}:', result[0].data.evs)

In [ ]:
IdealExpectation = []
for i in range (300):
    CirTran = q.compiler.transpile(testCir, backend=backend, optimization_level=0)
    job = estimator.run([(CirTran, IdealObservable)])
    result = job.result()
    IdealExpectation.append(result[0].data.evs)
    #print(f'Expectation_{i + 1}:', result[0].data.evs)

In [ ]:
# Combine the two sets of expectations for histogram
all_expectations = ErrorExpectation + IdealExpectation

# Parameters for histogram
bin_width = 0.005
bins = np.arange(min(all_expectations), max(all_expectations) + bin_width, bin_width)

# Plotting the histogram
plt.figure(figsize=(10, 6))
plt.hist(ErrorExpectation, bins=bins, alpha=0.5, label='Error', color='blue', edgecolor='black')
plt.hist(IdealExpectation, bins=bins, alpha=0.5, label='Ideal', color='green', edgecolor='black')

# Calculate and plot statistics for set 1
mean_set1 = np.mean(ErrorExpectation)
median_set1 = np.median(ErrorExpectation)
variance_set1 = np.var(ErrorExpectation)
std_dev_set1 = np.std(ErrorExpectation)

# Calculate and plot statistics for set 2
mean_set2 = np.mean(IdealExpectation)
median_set2 = np.median(IdealExpectation)
variance_set2 = np.var(IdealExpectation)
std_dev_set2 = np.std(IdealExpectation)

# Annotating the statistics on the plot
plt.axvline(mean_set1, color='blue', linestyle='dashed', linewidth=1, label=f'Error: {mean_set1:.4f}')
plt.axvline(mean_set2, color='green', linestyle='dashed', linewidth=1, label=f'Ideal: {mean_set2:.4f}')

# Adding labels, title, and legend
plt.xlabel('Expectation Values')
plt.ylabel('Frequency')
plt.title('Histogram of Expectation Values (Bin width = 0.005)')

# Move the legend to the center
plt.legend(loc='upper center', fontsize=10)

# Display the statistics for both sets in the middle of the plot
plt.text((min(all_expectations) + max(all_expectations)) / 2 - 0.16, 37.5, 
         f'Median Set 1: {median_set1:.4f}\nVariance Set 1: {variance_set1:.4f}\nStd Dev Set 1: {std_dev_set1:.4f}', 
         color='blue', fontsize=10)
plt.text((min(all_expectations) + max(all_expectations)) / 2 + 0.04, 37.5, 
         f'Median Set 2: {median_set2:.4f}\nVariance Set 2: {variance_set2:.4f}\nStd Dev Set 2: {std_dev_set2:.4f}', 
         color='green', fontsize=10)

# Show the plot
plt.grid(True)
plt.show()